# 2025 - 2학기 머신러닝 팀플 - 제출용(scoring)
## 주제. Colored MNIST classification

#### 팀명: 러닝머신 🏃
- 20211513 김동준
- 20211546 이채현 (조장 🙋)
- 20212996 허준범


### cuML 사용시 NVIDA-GPU cuDNN 및 cuML 설치 되어있어야함
(colab 환경에서는 T4 GPU 인스턴스 사용)

# parameters (수정 필요)

In [ ]:
# cuML 사용 여부 설정
USE_CUML = True

# 모델 저장 경로 설정
cuml_model_directory = './trained/final_svc_bundle_cuml.joblib'
sklearn_model_directory = './trained/final_svc_bundle_sklearn.joblib'

# 데이터셋 경로 설정
test_data_directory = './input/fake_test_set.npz'

# Load test set (label 수정 필요)

In [ ]:
import numpy as np

# load test set
test_set = np.load(test_data_directory, allow_pickle = True)

# test_set
test_images = test_set["images"][-10000:] # 
test_labels = test_set["digit"][-10000:]  #
test_fg = test_set["foreground"][-10000:] #
test_bg = test_set["background"][-10000:] #

# 무지개 색상 팔레트 (RGB)
RAINBOW_COLORS_NAME_RGB = {
    'RED' : (255, 0, 0),
    'ORANGE' : (255, 127, 0),
    'YELLOW' : (255, 255, 0),
    'GREEN' : (0, 255, 0),
    'BLUE' : (0, 0, 255),
    'INDIGO' : (75, 0, 130),
    'VIOLET' : (148, 0, 211)
}

In [ ]:
import matplotlib.pyplot as plt
image = test_images[0]
if image is not None:
    # 생성된 이미지를 화면에 표시합니다 (cmap='gray'는 값이 낮을수록 어둡게 표시).
    plt.imshow(image, cmap='gray')
    plt.axis('off')
    plt.title(f"NUM={digit}, FONT={os.path.basename(font_path)}")

    plt.tight_layout()
    plt.show()

# Load trained model

In [ ]:
import joblib

if USE_CUML:
    from cuml.svm import SVC as cu_SVC
    from cuml.multiclass import OneVsOneClassifier as cu_OneVsOneClassifier
    import cupy as cp
    # --- 로드 및 재설정 ---
    # 로드 후, 반드시 gamma를 수동으로 강제 적용해야 합니다.
    loaded_bundle = joblib.load(cuml_model_directory)
    loaded_ovo_model = loaded_bundle['ovo_model']
    loaded_gamma = loaded_bundle['gamma']

    # 🚨 gamma 값 강제 재설정 (핵심)
    if hasattr(loaded_ovo_model, 'estimators_'):
        for estimator in loaded_ovo_model.estimators_:
            estimator.gamma = loaded_gamma

else:
    from sklearn.svm import SVC as sk_SVC
    from sklearn.multiclass import OneVsOneClassifier as sk_OneVsOneClassifier
    loaded_bundle = joblib.load(sklearn_model_directory)
    loaded_ovo_model = loaded_bundle['ovo_model']
    loaded_gamma = loaded_bundle['gamma']

    # 🚨 gamma 값 강제 재설정 (핵심)
    if hasattr(loaded_ovo_model, 'estimators_'):
        for estimator in loaded_ovo_model.estimators_:
            estimator.gamma = loaded_gamma

In [ ]:
import gc
# 이제 OneVsOneClassifier_svc_class를 사용하여 예측 수행 가능

def predict_digit(converted_images):
    print("Predicting using the loaded OneVsOneClassifier model...")
    # 이미지 데이터를 2D 배열로 변환
    X_test = cp.asarray(converted_images.reshape(-1, 28*28).astype(np.float32)) if USE_CUML else converted_images.reshape(-1, 28*28).astype(np.float32)

    # 예측 수행
    predicted_labels = loaded_ovo_model.predict(X_test)

    predicted_labels_cpu = predicted_labels.get() if USE_CUML else predicted_labels
    del predicted_labels
    gc.collect()

    return predicted_labels_cpu

# color 분류 함수 정의

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- 필요한 상수 정의 (루프 밖) ---
PALETTE_NAMES = list(RAINBOW_COLORS_NAME_RGB.keys())
PALETTE_RGBS = np.array(list(RAINBOW_COLORS_NAME_RGB.values()), dtype=np.float32)

# --- 새로운 리스트로 이미지 저장 ---
bw_images_list = [] 
fg_color_names = []
bg_color_names = []

def predict_colors(rgb_array):
    for i, colored_image in enumerate(tqdm(rgb_array)): # 인덱스 i를 추가하여 레이블 접근
        H, W, C = colored_image.shape
        pixels_float = colored_image.reshape(-1, 3).astype(np.float32)

        # 1. 🚀 벡터화된 거리 계산 (가장 가까운 색상 인덱스 찾기)
        distances = np.sum((pixels_float[:, np.newaxis, :] - PALETTE_RGBS)**2, axis=2)
        closest_indices = np.argmin(distances, axis=1)

        # 2. 🛡️ 노이즈에 강한 빈도수 계산
        palette_color_counts = {name: 0 for name in PALETTE_NAMES}
        unique_indices, counts = np.unique(closest_indices, return_counts=True)
        
        for index, count in zip(unique_indices, counts):
            palette_color_counts[PALETTE_NAMES[index]] = count

        # 3. 전경/배경 결정
        found_colors_array = sorted(palette_color_counts.items(), key=lambda x: x[1], reverse=True)
        
        if len(found_colors_array) < 2:
            bw_images_list.append(np.zeros((H, W), dtype=np.uint8))
            continue
        
        # 찾은 전경/배경 색상 이름
        # 1순위 (가장 빈도 높음)
        found_fg_name = found_colors_array[1][0] 
        # 2순위
        found_bg_name = found_colors_array[0][0] 
        # print(f"{filenames[i]}, predic: {found_fg_name}, {found_bg_name}, actual: {fg[i]}, {bg[i]}")

        fg_color_names.append(found_fg_name)
        bg_color_names.append(found_bg_name)
        
        
        # ----------------------------------------------------

        # 4. 🚀 마스크 생성 (기존 로직 유지)
        target_fg_index = PALETTE_NAMES.index(found_fg_name)
        match_mask = (closest_indices == target_fg_index).reshape(H, W) 
        
        mask_image = np.zeros(match_mask.shape, dtype=np.uint8)
        mask_image[match_mask] = 255
        
        # 5. 리스트에 추가
        bw_images_list.append(np.array(mask_image))

    # 최종 변환
    return np.array(bw_images_list), np.array(fg_color_names), np.array(bg_color_names)

# 실제 추론

In [ ]:
bw_images, predicted_fg, predicted_bg = predict_colors(test_images)
predicted_digit = predict_digit(bw_images)

# scoring

In [ ]:
from sklearn.metrics import accuracy_score as sk_accuracy_score

digit_accuracy = sk_accuracy_score(test_labels, predicted_digit)
fg_accuracy = sk_accuracy_score(test_fg, predicted_fg)
bg_accuracy = sk_accuracy_score(test_bg, predicted_bg)

print(f"Digit Accuracy: {digit_accuracy * 100:.4f}%")
print(f"Foreground Color Accuracy: {fg_accuracy * 100:.4f}%")
print(f"Background Color Accuracy: {bg_accuracy * 100:.4f}%")